<p><font size="6" color='grey'> <b>
KI-Agenten. Verstehen. Anwenden. Gestalten.
</b></font> </br></p>



<p><font size="5" color='grey'> <b>
DeepAgents - Meeting/Briefing-Skill_
</b></font> </br></p>

---

In [ ]:
#@title 🔧 Umgebung einrichten{ display-mode: "form" }
!uv pip install --system -q git+https://github.com/ralf-42/Agenten.git#subdirectory=04_modul

import json
import os
import subprocess
import sys
import tempfile
import time
import urllib.request
from pathlib import Path
from textwrap import dedent
from typing import List, Optional

from langchain.chat_models import init_chat_model
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.tools import tool
from markitdown import MarkItDown
from pydantic import BaseModel, Field
from deepagents import create_deep_agent

from genai_lib.utilities import (
    check_environment, get_ipinfo, mprint, mermaid, setup_api_keys, show_trace,
)

os.environ["LANGSMITH_TRACING"]  = "true"
os.environ["LANGSMITH_PROJECT"]  = "M33-Meeting-Briefing"
os.environ["LANGSMITH_ENDPOINT"] = "https://eu.api.smith.langchain.com"

setup_api_keys(["OPENAI_API_KEY", "LANGSMITH_API_KEY"], create_globals=False)
print()
check_environment()
print()
get_ipinfo()

In [ ]:
#@title 📦 Weitere Pakete { display-mode: "form" }

!uv pip install --system -q "deepagents==0.4.12"
!uv pip install --system -q markitdown[all]

# 1 | Übersicht

---

<div style="background-color:#1a1a2e; padding:30px; border-radius:12px; margin-bottom:10px">
  <h1 style="color:#e0e0e0; font-family:monospace; margin:0">Meeting-Briefing Skill mit DeepAgents</h1>
  <p style="color:#a0a0c0; font-family:monospace; margin:8px 0 0 0">Skill-Dateien aus GitHub · Gate-Analyse mit o3 · structured_output · Writer Sub-Agent</p>
</div>

**Was dieses Notebook zeigt:**
- Skill-Dateien (`SKILL.md`, `WRITER.md`, Regelwerke) **direkt aus dem GitHub-Repo laden**
- Gate-Analyse mit `o3` + `with_structured_output()` → deterministisches JSON
- `extract_actions.py` aus GitHub holen, lokal zwischenspeichern, deterministisch aufrufen
- **Writer als Sub-Agent** mit geladenem `WRITER.md` als System-Prompt
- DeepAgent-Koordinator orchestriert Skill-Abarbeitung autonom

**Verwandte Module:** M31 – Agent Skill Compliance · M32 – DeepAgents Harness

---

<font color="black" size="5">Was dieses Notebook zeigt</font>

In `06_skill/meeting-briefing` im GitHub-Repo liegt ein vollständiger Skill:
Kontext laden, Agenda nach festen Regeln strukturieren, Action Items deterministisch
extrahieren und das Ergebnis im definierten Briefing-Format ausgeben.

Dieses Notebook baut daraus einen **DeepAgent** mit vier Kernmerkmalen:

| # | Merkmal | Warum |
|---|---------|-------|
| 1 | **Skill-Dateien aus GitHub laden** | Änderungen am Skill greifen ohne Notebook-Anpassung |
| 2 | **`with_structured_output()`** | Gate-Analyse gibt Pydantic-validiertes JSON zurück |
| 3 | **Modell-Auswahl laut Guide** | `o3` für Koordinator & Gate-Analyse |
| 4 | **Writer als Sub-Agent** | Trennung von Analyse und Formatierung |

> **Best Practice:** DeepAgents ist hier die Harness-Schicht, nicht die Wissensquelle.
> Die Fachlogik bleibt im Skill-Repo, in den Regelwerken und im deterministischen Tooling verankert.

In [ ]:
#@markdown   <p><font size="4" color="green">Skill-Architektur</font></p>

diagram_arch = '''
%%{init: {'theme':'light'}}%%
flowchart TB
    GH([" 🐙 GitHub Repo\nralf-42/Agenten"]) --> LOAD
    U([" 🧑 Meeting-Anfrage"]) --> K

    subgraph K_BLOCK [" 🤖 DeepAgent Koordinator — o3"]
        LOAD["Skill-Dateien laden\n(urllib.request)"] --> K
        K["Kontext laden\nAction Items extrahieren"]
        K --> G["🛠️ analysiere_meeting_kontext\no3 + with_structured_output"]
        K --> E["🛠️ extract_action_items\nextract_actions.py als Subprocess"]
        G --> JSON["GateOutput JSON\nPydantic-validiert"]
        E --> JSON
        JSON --> W["Sub-Agent\nbriefing-writer"]
    end

    W --> B([" ✅ Meeting-Briefing"])

    style GH   fill:#24292e,stroke:#555,color:#fff
    style U    fill:#10a37f,stroke:#333,color:#000
    style B    fill:#10a37f,stroke:#333,color:#000
    style G    fill:#87CEEB,stroke:#333,color:#000
    style E    fill:#87CEEB,stroke:#333,color:#000
    style JSON fill:#FFD700,stroke:#333,color:#000
    style W    fill:#FFA500,stroke:#333,color:#000
    style LOAD fill:#6e40c9,stroke:#333,color:#fff
'''
mermaid(diagram_arch, width=580)

# 2 | Skill-Dateien aus GitHub laden

---



Alle Skill-Dateien werden **zur Laufzeit aus dem GitHub-Repo** geholt:

| Datei | Zweck |
|-------|-------|
| `SKILL.md` | Gate-Regeln, Hard Rules, Workflow-Definition |
| `WRITER.md` | System-Prompt für den Writer Sub-Agenten |
| `references/agenda_rules.md` | Prioritäten, Agenda-Aufbau |
| `references/action_rules.md` | Action-Item-Erkennungsregeln |
| `references/examples.md` | Referenzbeispiele für Gate und Writer |
| `scripts/extract_actions.py` | Deterministisches Skript → Temp-Datei |

> 💡 Das Skript `extract_actions.py` wird in eine **Temp-Datei** geschrieben,
> damit es per `subprocess` aufgerufen werden kann — identisch zum Produktionseinsatz.

In [ ]:
# 2.1 Skill-Dateien aus GitHub-Repo laden

GITHUB_RAW  = "https://raw.githubusercontent.com/ralf-42/Agenten/main"
SKILL_BASE  = f"{GITHUB_RAW}/06_skill/meeting-briefing"

SKILL_URLS = {
    "skill":        f"{SKILL_BASE}/SKILL.md",
    "writer":       f"{SKILL_BASE}/WRITER.md",
    "agenda_rules": f"{SKILL_BASE}/references/agenda_rules.md",
    "action_rules": f"{SKILL_BASE}/references/action_rules.md",
    "examples":     f"{SKILL_BASE}/references/examples.md",
}
SCRIPT_URL  = f"{SKILL_BASE}/scripts/extract_actions.py"


def fetch_text(url: str) -> str:
    """Lädt eine Text-Datei per HTTP von GitHub."""
    with urllib.request.urlopen(url) as resp:
        return resp.read().decode("utf-8").strip()


# Einmalig laden und cachen
_skill_cache: dict[str, str] = {}

def read_skill(name: str) -> str:
    """Gibt den Inhalt einer Skill-Datei zurück (mit Cache)."""
    if name not in _skill_cache:
        _skill_cache[name] = fetch_text(SKILL_URLS[name])
    return _skill_cache[name]


# extract_actions.py als Temp-Datei bereitstellen
_script_src  = fetch_text(SCRIPT_URL)
SCRIPT_PATH  = Path(tempfile.gettempdir()) / "extract_actions_m36.py"
SCRIPT_PATH.write_text(_script_src, encoding="utf-8")

mprint(f"GitHub-Basis: `{SKILL_BASE}`")
mprint(f"Skill-Dateien: {list(SKILL_URLS.keys())}")
mprint(f"Skript-Cache: `{SCRIPT_PATH}`")

In [ ]:
# 2.2 Demo-Kontext-Dokumente aus GitHub laden

DATA_BASE = f"{GITHUB_RAW}/02_daten/01_text"

_VORB_FILES = {
    "01_meeting_auftrag.docx": f"{DATA_BASE}/01_meeting_auftrag.docx",
    "02_projektstatus.docx":   f"{DATA_BASE}/02_projektstatus.docx",
    "03_mail_verlauf.docx":    f"{DATA_BASE}/03_mail_verlauf.docx",
    "04_review_notizen.docx":  f"{DATA_BASE}/04_review_notizen.docx",
}

_md = MarkItDown()

DEMO_CONTEXTS_VORB = {
    name: _md.convert(url).text_content
    for name, url in _VORB_FILES.items()
}

# Nachbereitung: Kundengespräch (inline)
DEMO_CONTEXTS_NACH = {
    "gespraechsprotokoll.md": dedent("""
        Gesprächsprotokoll — Abstimmung Pilotprojekt Firma XYZ, 2026-03-19

        Entschieden: Pilotprojekt startet 01.04.2026, Scope 3 Use Cases, Laufzeit 8 Wochen.
        Ansprechpartner Kunde: Dr. Mueller.
        Petra sendet Projektvertrag an Dr. Mueller bis 2026-03-21.
        Ralf abstimmt Kick-off-Termin bis 2026-03-25.
        Ralf erstellt Use-Case-Dokumentation bis 2026-03-28.
    """).strip(),
    "offene_punkte.txt": dedent("""
        Offene Punkte nach Kundengespräch:

        Budgetfreigabe seitens XYZ noch ausstehend — kein Datum bekannt.
        Technische Infrastruktur beim Kunden muss noch geprueft werden.
    """).strip(),
}

mprint(f"Vorbereitung: {list(DEMO_CONTEXTS_VORB.keys())}")
mprint(f"Nachbereitung: {list(DEMO_CONTEXTS_NACH.keys())}")

In [ ]:
from genai_lib.utilities import load_prompt

# Auszüge aus den GitHub-Skill-Dateien (Frontmatter wird automatisch entfernt)
_previews = {
    "SKILL.md":        (SKILL_URLS["skill"],        600),
    "agenda_rules.md": (SKILL_URLS["agenda_rules"], 400),
    "WRITER.md":       (SKILL_URLS["writer"],        300),
}

zeilen = []
for name, (url, limit) in _previews.items():
    text = load_prompt(url, mode="S")[:limit]
    zeilen += [f"### {name} (Auszug)", "", text, ""]

mprint("\n".join(zeilen))

# 3 | Pydantic-Modelle & Gate-Analyse

---



Damit der Gate-Agent **deterministisch** strukturierten Output liefert,
verwenden wir `with_structured_output()` mit Pydantic-Modellen.

| Modell | Aufgabe |
|--------|---------|
| `ActionItemSchema` | Einzelnes Action Item (Was / Wer / Wann) |
| `AgendaPoint` | Agenda-Punkt (Priorität, Ziel, Dauer, Eigentümer) |
| `GateOutput` | Vollständiger strukturierter Gate-Output für den Writer |

> 💡 **Warum `with_structured_output()`?**
> Der Writer-Sub-Agent braucht verlässliche Eingabedaten. Ein freier LLM-Text-Output
> kann in Struktur und Vollständigkeit variieren — ein Pydantic-validiertes JSON nicht.

In [ ]:
# 3.1 Pydantic-Modelle für strukturierten Gate-Output

class ActionItemSchema(BaseModel):
    """Ein konkretes Action Item aus dem Meeting."""
    task:  str = Field(description="Aufgabenbeschreibung — kurz, aktionsorientiert")
    owner: str = Field(description="Verantwortliche Person — '[offen]' wenn unklar")
    due:   str = Field(description="Fälligkeitsdatum — '[offen]' wenn unbekannt")


class AgendaPoint(BaseModel):
    """Ein Agenda-Punkt mit Priorität, Ziel und Zeitplan."""
    priority: str = Field(description="Priorität: hoch, mittel oder gering")
    point:    str = Field(description="Bezeichnung des Agenda-Punkts")
    goal:     str = Field(description="Ziel: entscheiden, klären, informieren oder präsentieren")
    duration: int = Field(description="Geplante Dauer in Minuten")
    owner:    str = Field(description="Verantwortliche Person für diesen Punkt")


class GateOutput(BaseModel):
    """Strukturierter Gate-Output — Eingabe für den Writer-Sub-Agenten."""
    type:           str                    = Field(description="vorbereitung oder nachbereitung")
    meeting_type:   str                    = Field(description="Projektmeeting, Kundengespräch, Review, Workshop oder Sonstiges")
    topic:          str                    = Field(description="Meeting-Thema")
    date:           Optional[str]          = Field(default=None, description="Datum/Uhrzeit wenn bekannt")
    participants:   List[str]              = Field(description="Teilnehmerliste")
    agenda:         List[AgendaPoint]      = Field(description="Agenda-Punkte sortiert: hoch → mittel → gering")
    open_questions: List[str]              = Field(description="Offene Fragen vor dem Meeting")
    action_items:   List[ActionItemSchema] = Field(description="Action Items aus Kontext und Vorgesprächen")
    risks:          List[str]              = Field(description="Bekannte Risiken und Konflikte")
    status:         str                    = Field(description="ok, no_context oder conflict")
    conflict:       Optional[bool]         = Field(default=False, description="True bei widersprüchlichen Quellinformationen")
    decisions:      Optional[List[str]]    = Field(default=None, description="Nur Nachbereitung: getroffene Entscheidungen")
    open_points:    Optional[List[str]]    = Field(default=None, description="Nur Nachbereitung: vertagte Punkte")


# Gate-Prompt aus GitHub-Skill-Dateien zusammensetzen
_GATE_RULES = (
    read_skill("skill") + "\n\n"
    + read_skill("agenda_rules") + "\n\n"
    + read_skill("action_rules")
)

gate_prompt = ChatPromptTemplate.from_messages([
    ("system", _GATE_RULES),
    ("human", "Meeting-Anfrage: {anfrage}\n\nKontext-Dokumente:\n{kontext}"),
])

mprint("✅ **Pydantic-Modelle** definiert: `ActionItemSchema`, `AgendaPoint`, `GateOutput`")
mprint("✅ **Gate-Prompt** aus GitHub: `SKILL.md` + `agenda_rules.md` + `action_rules.md`")

In [ ]:
# 3.2 Gate-Analyse als @tool — o3 + with_structured_output()

@tool
def analysiere_meeting_kontext(anfrage: str, kontext: str) -> str:
    """
    Analysiert Meeting-Kontext mit o3 und gibt strukturierten Gate-Output zurück.

    Verwendet with_structured_output(GateOutput) für deterministisches,
    Pydantic-validiertes JSON. Regeln kommen aus den GitHub-Skill-Dateien.

    Args:
        anfrage: Meeting-Typ, Thema, Teilnehmer, Datum
        kontext: Gesamter Meeting-Kontext aus allen geladenen Dokumenten

    Returns:
        Pydantic-validiertes JSON (GateOutput) als String
    """
    gate_llm   = init_chat_model("openai:o3")   # kein temperature!
    gate_chain = gate_prompt | gate_llm.with_structured_output(GateOutput)
    result: GateOutput = gate_chain.invoke({"anfrage": anfrage, "kontext": kontext})
    return result.model_dump_json(indent=2)


mprint("✅ **`analysiere_meeting_kontext`** — o3 + `with_structured_output(GateOutput)`")

# 4 | Skill-Tools kapseln

---



Der Koordinator bekommt sechs fokussierte Tools:

| Tool | Quelle | Aufgabe |
|------|--------|---------|
| `list_context_documents` | Notebook | Verfügbare Demo-Dokumente anzeigen |
| `load_context_document` | Notebook | Einzelnes Kontextdokument laden |
| `load_skill_asset` | **GitHub** | Skill-Regelwerk zur Laufzeit nachladen |
| `extract_action_items` | **GitHub** (Temp) | Deterministisch via `extract_actions.py` |
| `analysiere_meeting_kontext` | LLM-Chain | o3 + `with_structured_output(GateOutput)` |
| `meeting_request` | Notebook | Demo-Anfrage für Beispiel 1 |

In [ ]:
# 4.1 Tool-Definitionen

@tool
def list_context_documents(typ: str = "vorbereitung") -> str:
    """Listet verfügbare Demo-Kontextdokumente für den angegebenen Meeting-Typ.

    Args:
        typ: 'vorbereitung' oder 'nachbereitung'
    """
    docs = DEMO_CONTEXTS_VORB if typ == "vorbereitung" else DEMO_CONTEXTS_NACH
    return json.dumps(sorted(docs.keys()), ensure_ascii=False)


@tool
def load_context_document(name: str, typ: str = "vorbereitung") -> str:
    """Lädt ein einzelnes Demo-Kontextdokument.

    Args:
        name: Dateiname (aus list_context_documents)
        typ:  'vorbereitung' oder 'nachbereitung'
    """
    docs = DEMO_CONTEXTS_VORB if typ == "vorbereitung" else DEMO_CONTEXTS_NACH
    if name not in docs:
        return f"Dokument '{name}' nicht gefunden. Verfügbar: {sorted(docs)}"
    return docs[name]


@tool
def load_skill_asset(name: str) -> str:
    """Lädt eine Skill-Datei aus dem GitHub-Repo (SKILL.md, WRITER.md, Regelwerke, Beispiele).

    Erlaubte Werte: skill, writer, agenda_rules, action_rules, examples

    Quelle: github.com/ralf-42/Agenten — 06_skill/meeting-briefing/
    """
    key = name.strip().lower()
    if key not in SKILL_URLS:
        return f"Unbekanntes Asset '{name}'. Verfügbar: {list(SKILL_URLS.keys())}"
    return read_skill(key)   # aus Cache oder frisch von GitHub


@tool
def extract_action_items(text: str) -> str:
    """Extrahiert Action Items deterministisch über extract_actions.py (aus GitHub, lokal gecacht).

    Erkennt Aktionsverben, Eigentümer und Fälligkeitsdaten.
    Gibt JSON mit top 5 priorisierten Action Items + Backlog zurück.
    """
    payload = json.dumps({"text": text}, ensure_ascii=False)
    proc = subprocess.run(
        [sys.executable, str(SCRIPT_PATH), payload],
        capture_output=True, text=True, encoding="utf-8", check=False,
    )
    if proc.returncode != 0:
        return json.dumps({
            "status": "error",
            "stderr": proc.stderr.strip(),
            "stdout": proc.stdout.strip(),
        }, ensure_ascii=False, indent=2)
    return proc.stdout.strip()


@tool
def meeting_request() -> str:
    """Liefert die Demo-Meeting-Anfrage für Beispiel 1 (Vorbereitung Sprint-Review)."""
    return dedent("""
        Thema: Sprint-Review Meeting-Briefing Skill Modul M36
        Typ: Vorbereitung
        Datum/Zeit: 2026-03-27 10:00
        Teilnehmer: Lea (Produkt), Murat (Engineering), Sophie (Training), Jonas (QA), Klara (Moderation)
        Dauer: 30 Minuten
        Ziel: Meeting-Briefing fuer interne Review-Runde vorbereiten.
    """).strip()


ALLE_TOOLS = [
    meeting_request,
    list_context_documents,
    load_context_document,
    load_skill_asset,
    extract_action_items,
    analysiere_meeting_kontext,
]

mprint(f"✅ **{len(ALLE_TOOLS)} Tools** registriert: {[t.name for t in ALLE_TOOLS]}")

In [ ]:
# 4.2 extract_action_items — Smoke-Test
kombinierter_kontext = "\n\n".join(
    [f"## {name}\n{text}" for name, text in DEMO_CONTEXTS_VORB.items()]
)

raw = extract_action_items.invoke({"text": kombinierter_kontext})
result = json.loads(raw)

zeilen = [
    "### 🔍 Extrahierte Action Items (Vorbereitung-Kontext)",
    "",
    "| # | Aufgabe | Verantwortlich | Fällig |",
    "|---|---------|----------------|--------|",
]
for i, item in enumerate(result.get("action_items", []), 1):
    zeilen.append(f'| {i} | {item["task"]} | {item["owner"]} | {item["due"]} |')
if result.get("backlog_count", 0) > 0:
    zeilen.append(f'\n> ⚠️ **Backlog:** {result["backlog_count"]} weitere Item(s)')
mprint("\n".join(zeilen))

<font color="black" size="5">Best-Practice-Hinweise für dieses Beispiel</font>

- **Version pinnen:** `deepagents==0.4.12` — die `subagents=[...]`-Syntax ist noch nicht stabil.
- **Skill-Dateien aus GitHub:** Regeländerungen in `SKILL.md` wirken beim nächsten Notebook-Run
  automatisch — kein manuelles Copy-Paste in Prompts.
- **Cache für GitHub-Requests:** `read_skill()` lädt jede Datei nur einmal pro Session.
  Für Entwicklung: `_skill_cache.clear()` erzwingt ein Neuladen.
- **Skript als Temp-Datei:** `extract_actions.py` wird beim Setup in `tempfile.gettempdir()`
  geschrieben. Kein permanentes File-Schreiben — sauber für Cloud-Umgebungen.
- **`recursion_limit` großzügig setzen:** Planning, Tool-Calls und Sub-Agent-Delegation
  zählen alle als eigene Schritte.
- **Modell-Hinweis:** Sub-Agenten erben das Modell des Koordinators (`o3`). In einer
  Produktionsumgebung würde man den Writer mit `gpt-5.1` betreiben, sobald
  DeepAgents ein `model`-Feld pro Sub-Agent unterstützt.

# 5 | DeepAgent aufbauen

---

In [ ]:
# 5.1 Writer Sub-Agent — System-Prompt direkt aus GitHub (WRITER.md)
writer_subagent = {
    "name": "briefing-writer",
    "description": "Formatiert den Gate-Output exakt im Ausgabeformat des Meeting-Briefing-Skills.",
    "system_prompt": (
        read_skill("writer")           # WRITER.md live aus GitHub
        + "\n\n"
        + "Gib ausschließlich das fertige Briefing aus.\n"
        + "Keine Websuche, keine neuen Fakten, keine Spekulation."
    ),
    "tools": [],
}

# 5.2 DeepAgent Koordinator
KOORDINATOR_PROMPT = (
    "Du bist der Koordinator fuer den meeting-briefing Skill.\n\n"
    "Pflicht-Ablauf:\n"
    "1. meeting_request aufrufen (Beispiel 1) ODER Anfrage aus Nachricht lesen\n"
    "2. list_context_documents aufrufen und ALLE Dokumente mit load_context_document laden\n"
    "3. extract_action_items fuer den gesamten geladenen Kontext aufrufen\n"
    "4. analysiere_meeting_kontext aufrufen → GateOutput JSON erhalten\n"
    "5. GateOutput an briefing-writer delegieren zur finalen Formatierung\n\n"
    "Hard Rules:\n"
    "- IMMER extract_action_items aufrufen — nie Action Items frei erfinden\n"
    "- IMMER analysiere_meeting_kontext aufrufen — kein freies Briefing-Schreiben\n"
    "- Fehlende Angaben als [offen] markieren, nie spekulieren\n"
    "- Antwort auf Deutsch"
)

meeting_agent = create_deep_agent(
    model=init_chat_model("openai:o3-mini"),   # Koordinator → o3-mini (Orchestrator, kein kritischer Judge)
    tools=ALLE_TOOLS,
    subagents=[writer_subagent],
    system_prompt=KOORDINATOR_PROMPT,
)

mprint("✅ **DeepAgent** erstellt")
mprint(f"   Koordinator: `o3-mini`  ·  Sub-Agent: `briefing-writer`  ·  Tools: {len(ALLE_TOOLS)}")

In [ ]:
#@markdown   <p><font size="4" color='green'>  Ablauf im Koordinator</font> </br></p>


# Sequenzdiagramm: Ablauf im Koordinator
diagram_seq = '''
%%{init: {'theme':'light'}}%%
sequenceDiagram
    autonumber
    actor N as Nutzer
    participant K as Koordinator (o3)
    participant GH as GitHub Repo
    participant D as Dokument-Tools
    participant G as analysiere_meeting_kontext
    participant E as extract_action_items
    participant W as briefing-writer

    Note over K,GH: Setup (einmalig)
    GH-->>K: SKILL.md, WRITER.md, Regelwerke, extract_actions.py

    N->>K: Meeting-Anfrage
    K->>D: list_context_documents()
    D-->>K: ["doc1.md", "doc2.txt", ...]
    K->>D: load_context_document(name) [x n]
    D-->>K: Dokumentinhalte
    K->>E: extract_action_items(kontext)
    E-->>K: Action Items JSON
    K->>G: analysiere_meeting_kontext(anfrage, kontext)
    G-->>K: GateOutput JSON (Pydantic-validiert)
    K->>W: GateOutput übergeben
    W-->>K: Formatiertes Briefing
    K-->>N: Meeting-Briefing
'''
mermaid(diagram_seq, width=1000)

In [ ]:
# -- VIZ --
from IPython.display import Image, display

display(Image(meeting_agent.get_graph(xray=True).draw_mermaid_png()))

# 6 | Beispiel-Runs

---

| Beispiel | Typ | Szenario |
|----------|-----|----------|
| 1️⃣ | Vorbereitung | Sprint-Review Modul M36 — via Demo-Dokument-Tools |
| 2️⃣ | Nachbereitung | Kundengespräch Pilotprojekt XYZ — Kontext direkt übergeben |

In [ ]:
import re as _re

def invoke_with_retry(agent, inputs, config, max_retries: int = 5):
    """Wiederholt agent.invoke() bei RateLimitError mit automatischer Wartezeit."""
    for attempt in range(1, max_retries + 1):
        try:
            return agent.invoke(inputs, config=config)
        except Exception as e:
            msg = str(e)
            if "rate_limit_exceeded" not in msg and "429" not in msg:
                raise
            match = _re.search(r"try again in ([\d.]+)s", msg)
            wait = float(match.group(1)) + 2 if match else 15
            print(f"Rate limit (Versuch {attempt}/{max_retries}) — warte {wait:.0f}s …")
            time.sleep(wait)
    raise RuntimeError(f"Rate limit nach {max_retries} Versuchen nicht behoben.")

In [ ]:
# 6.1 Beispiel: Sprint-Review Vorbereitung
aufgabe_1 = dedent("""
    Bereite das Meeting vor und wende den meeting-briefing Skill korrekt an.
    Nutze alle verfuegbaren Demo-Dokumente (typ='vorbereitung'),
    extrahiere Action Items mit dem Tool und gib am Ende nur das fertige Briefing aus.
""").strip()

print(aufgabe_1)
print()

t0 = time.perf_counter()
result_1 = invoke_with_retry(
    meeting_agent,
    {"messages": [{"role": "user", "content": aufgabe_1}]},
    config={
        "recursion_limit": 120,
        "run_name": "m33-briefing-vorbereitung",
        "tags": ["m36", "deepagents", "skill", "vorbereitung"],
    },
)
latenz_1 = round((time.perf_counter() - t0) * 1000)

mprint("\n".join([
    "## Laufmetriken Beispiel 1",
    "",
    "| Metrik | Wert |",
    "|--------|------|",
    f"| Nachrichten gesamt | `{len(result_1['messages'])}` |",
    f"| Latenz             | `{latenz_1} ms` |",
    f"| Letzte Rolle       | `{result_1['messages'][-1].type}` |",
]))

In [ ]:
briefing_1 = result_1["messages"][-1].content
mprint(briefing_1)

In [ ]:
# 6.2 Beispiel: Kundengespräch Nachbereitung
kontext_nach = "\n\n".join(
    [f"## {name}\n{text}" for name, text in DEMO_CONTEXTS_NACH.items()]
)

aufgabe_2 = dedent(f"""
    Erstelle die Nachbereitung fuer das folgende Kundengespräch.
    Wende den meeting-briefing Skill korrekt an und extrahiere
    Action Items mit dem Tool. Gib am Ende nur das fertige Briefing aus.

    Meeting-Anfrage:
    - Thema: Abstimmung Pilotprojekt Firma XYZ
    - Datum: 2026-03-19
    - Teilnehmer: Ralf (intern), Dr. Mueller (Kunde), Petra (Vertrieb)
    - Typ: Nachbereitung

    Kontextdokumente:
    {kontext_nach}
""").strip()

t0 = time.perf_counter()
result_2 = invoke_with_retry(
    meeting_agent,
    {"messages": [{"role": "user", "content": aufgabe_2}]},
    config={
        "recursion_limit": 120,
        "run_name": "m33-briefing-nachbereitung",
        "tags": ["m36", "deepagents", "skill", "nachbereitung"],
    },
)
latenz_2 = round((time.perf_counter() - t0) * 1000)

mprint("\n".join([
    "## Laufmetriken Beispiel 2",
    "",
    "| Metrik | Wert |",
    "|--------|------|",
    f"| Nachrichten gesamt | `{len(result_2['messages'])}` |",
    f"| Latenz             | `{latenz_2} ms` |",
    f"| Letzte Rolle       | `{result_2['messages'][-1].type}` |",
]))

briefing_2 = result_2["messages"][-1].content
mprint(briefing_2)

In [ ]:
#@markdown   <p><font size="4" color="green">Trace-Analyse</font></p>

time.sleep(2)
show_trace("M33-Meeting-Briefing", limit=3, show_steps=True)

# 7 | Einordnung

---

<font color="black" size="5">Was an diesem Beispiel wichtig ist</font>

**Skill-Dateien aus dem Repo — nicht im Notebook**

Prompts, Regeln und Skripte kommen aus `github.com/ralf-42/Agenten`.
Das Notebook ist der Ausführungskontext, nicht die Wissensquelle.
Ändert sich `SKILL.md` im Repo, übernimmt das Notebook die neue Version
beim nächsten Run automatisch — ohne manuelles Copy-Paste.

**Structured Output schützt den Writer**

`with_structured_output(GateOutput)` stellt sicher, dass der Writer-Sub-Agent
immer vollständige, typisierte Eingaben erhält — unabhängig davon, wie der
Koordinator die Analyse formuliert.

**Grenzen des Musters**

| Grenze | Erklärung |
|--------|-----------|
| Sub-Agent erbt Modell | Writer nutzt `o3` statt idealerweise `gpt-5.1` (DeepAgents-Limit) |
| Kein HITL | Für Produktion: `interrupt()` vor Briefing-Versand ergänzen (→ M17) |
| Kein Judge | Skill-Compliance nicht automatisch geprüft (→ M31) |
| Kein Auth | GitHub-Zugriff anonym — für private Repos: Token-Authentifizierung ergänzen |

# A | Aufgaben

---



<font color="black" size="5">**Aufgabe 1: Eigenes Meeting-Briefing**</font>

Erstellt ein Briefing für ein eigenes Szenario:
- Fügt ein neues Demo-Dokument zu `DEMO_CONTEXTS_VORB` hinzu
- Wählt Thema, Teilnehmer und Meeting-Typ frei
- Führt den Agenten aus und prüft ob die Agenda-Prioritäten stimmen

---



<font color="black" size="5">**Aufgabe 2: no_context-Fall testen**</font>

Testet den Sonderfall `status: "no_context"`:
- Ruft `analysiere_meeting_kontext.invoke({"anfrage": "Teammeeting", "kontext": ""})`
- Prüft ob `status == "no_context"` im zurückgegebenen JSON gesetzt wird
- Leitet das JSON manuell an `briefing-writer` weiter
- Erwartetes Ergebnis: leeres Template mit Hinweis auf fehlenden Kontext



---

<font color="black" size="5">**Aufgabe 3 (Vertiefung): Konflikt-Szenario + Judge**</font>

Schritt 1 — Konflikt provozieren:
- Erstellt einen `kontext` mit zwei widersprüchlichen Budgetzahlen (120k€ vs. 95k€)
- Prüft ob `conflict: true` im GateOutput gesetzt wird
- Vergleicht mit Beispiel 4 aus `load_skill_asset("examples")`

Schritt 2 — Judge ergänzen:
- Baut einen LCEL-Judge-Step ein: `o3` + `with_structured_output(ComplianceCheck)`
- `ComplianceCheck` prüft ob alle Hard Rules aus `SKILL.md` eingehalten wurden
- Hard-Rules aus GitHub laden: `read_skill("skill")`
- Inspiriert durch: M24 – Agent Evaluation · M31 – Agent Skill Compliance

---

🎯 **Ziel:** Den Unterschied zwischen Skill-as-Code (im Repo versioniert) und
Skill-as-Prompt (im Notebook hardcodiert) in der Praxis erleben.